In [7]:
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import re
from scipy.sparse import hstack
import datetime

# ----------------------
# Load model
# ----------------------
model = pickle.load(open("model.pkl", "rb"))
word_vectorizer = pickle.load(open("word_vectorizer.pkl", "rb"))
char_vectorizer = pickle.load(open("char_vectorizer.pkl", "rb"))

app = FastAPI()


from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ----------------------
# Schema
# ----------------------
class EmailRequest(BaseModel):
    email: str

# ----------------------
# Cleaning
# ----------------------
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text

# ----------------------
# Keyword severity
# ----------------------
keyword_weights = {
    "verify": 0.15,
    "account": 0.1,
    "password": 0.2,
    "urgent": 0.15,
    "click": 0.1,
    "login": 0.15,
    "bank": 0.1,
    "suspend": 0.2
}

def rule_engine(text):
    score = 0
    found = []

    for word, weight in keyword_weights.items():
        if word in text:
            score += weight
            found.append(word)

    return score, found

# ----------------------
# URL Analyzer
# ----------------------
def detect_urls(text):
    urls = re.findall(r'http\S+|www\S+', text)
    suspicious = []

    for url in urls:
        if any(x in url for x in ["login", "verify", "secure", "update"]):
            suspicious.append(url)

    return len(urls), suspicious

# ----------------------
# Risk Engine
# ----------------------
def calculate_risk(ml_conf, rule_score, url_count):
    url_score = min(url_count * 0.2, 0.5)

    final_score = (
        (ml_conf * 0.6) +
        (rule_score * 0.3) +
        (url_score * 0.1)
    )

    return min(final_score, 1.0)

# ----------------------
# Prediction
# ----------------------
def predict_email(email):
    original_email = email
    cleaned = clean_text(email)

    # ML
    X_word = word_vectorizer.transform([cleaned])
    X_char = char_vectorizer.transform([cleaned])
    X = hstack([X_word, X_char])

    pred = model.predict(X)[0]
    prob = model.predict_proba(X)[0]
    ml_conf = float(max(prob))

    # Rules
    rule_score, keywords = rule_engine(cleaned)

    # URL
    url_count, suspicious_urls = detect_urls(original_email)

    # Risk
    final_score = calculate_risk(ml_conf, rule_score, url_count)

    final_prediction = "phishing" if final_score > 0.5 else "safe"

    # Explanation
    reasons = []
    if keywords:
        reasons.append(f"Suspicious keywords: {keywords}")
    if suspicious_urls:
        reasons.append(f"Suspicious URLs detected")
    if ml_conf > 0.8:
        reasons.append("ML model strongly confident")

    result = {
        "prediction": final_prediction,
        "confidence": round(final_score, 3),
        "ml_confidence": round(ml_conf, 3),
        "rule_score": round(rule_score, 3),
        "url_count": url_count,
        "keywords": keywords,
        "suspicious_urls": suspicious_urls,
        "reasons": reasons
    }

    log_request(original_email, result)

    return result

# ----------------------
# Logging
# ----------------------
def log_request(email, result):
    with open("logs.txt", "a") as f:
        f.write(f"{datetime.datetime.now()} | {email} | {result}\n")

# ----------------------
# Routes
# ----------------------
@app.get("/")
def home():
    return {"message": "Phishing Detection API 🚀"}

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(data: EmailRequest):
    return predict_email(data.email)

In [6]:
!uvicorn main:app --reload

^C


In [8]:
predict(EmailRequest(email="verify your account now http://secure-login.xyz"))

{'prediction': 'phishing',
 'confidence': 1.0,
 'ml_confidence': 0.997,
 'rules_triggered': ['verify', 'account'],
 'suspicious_urls': ['http://secure-login.xyz'],
 'url_count': 1}